## 1. 安裝環境與匯入套件


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import shutil
import subprocess
import sys
from pathlib import Path

'''
Use Kaggle's working directory and CUDA compiler.
使用 Kaggle 的工作目錄與 CUDA 編譯器。
'''
ROOT = Path('/kaggle/working')
PROJECT = ROOT / 'yue2.cpp'
SERVER = PROJECT / 'build/yue-server'
NVCC = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'

'''
Verify that both T4 GPUs and the CUDA compiler are available.
確認兩張 T4 GPU 與 CUDA 編譯器都可使用。
'''
check = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, check=True,
)
assert len(check.stdout.strip().splitlines()) == 2, '請在 Kaggle 選擇 GPU T4 ×2。'
assert Path(NVCC).is_file(), '找不到 nvcc。'
print(check.stdout, flush=True)

'''
Install only the packages used by the notebook.
只安裝 Notebook 實際使用的套件。
'''
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'hf_xet', 'requests'],
    check=True,
)

'''
Clone the official source or safely fast-forward an existing checkout.
複製官方原始碼，或以 fast-forward 安全更新現有倉庫。
'''
if not (PROJECT / '.git').exists():

    '''
    Avoid overwriting a pre-existing non-Git folder.
    避免覆寫既有的非 Git 專案目錄。
    '''
    assert not PROJECT.exists(), f'既有目錄不是 Git 倉庫：{PROJECT}'

    '''
    Download the official native backend and its submodules.
    下載官方原生引擎及其子模組。
    '''
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--recurse-submodules',
         'https://github.com/ServeurpersoCom/yue2.cpp.git', str(PROJECT)],
        check=True,
    )
else:

    '''
    Check for local source modifications before updating.
    更新前先檢查有無本機原始碼修改。
    '''
    source_changes = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip()

    '''
    Preserve custom local changes instead of overwriting them.
    保留使用者修改，不覆寫本機資料。
    '''
    if not source_changes:

        '''
        Update to the latest upstream revision without creating a merge commit.
        以 fast-forward 同步上游最新版本，不產生合併提交。
        '''
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', '-q'], check=True)
    else:

        '''
        Explain why the existing source was not updated.
        顯示未更新現有原始碼的原因。
        '''
        print('YuE2 原始碼已有本機修改，為避免衝突未更新；請在新 Session 重新執行。')

'''
Repair incomplete submodules without discarding the repository.
補齊不完整的子模組，不重新下載整個專案。
'''
subprocess.run(
    ['git', '-C', str(PROJECT), 'submodule', 'update', '--init', '--recursive'],
    check=True,
)

'''
Set the maximum build concurrency; reduce to 2 or 4 if memory is tight.
設定最大平行編譯數；若記憶體不足，請改成 2 或 4。
'''
BUILD_JOBS = 4

'''
Track the source revision used to compile the CUDA server binary.
記錄用來編譯 CUDA Server 的原始碼版本。
'''
BUILD_STAMP = PROJECT / 'build/.kaggle_yue2_revision'

'''
Read the currently checked-out source revision.
讀取目前使用的原始碼 commit。
'''
CURRENT_REVISION = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()

'''
Recompile when the binary is absent or the upstream source has changed.
當執行檔不存在或來源版本變更時重新編譯。
'''
if not SERVER.is_file() or not BUILD_STAMP.is_file() or BUILD_STAMP.read_text().strip() != CURRENT_REVISION:

    '''
    Honor the requested concurrency without claiming GPUs accelerate compilation.
    依指定平行數編譯；GPU 數量不會直接提升編譯速度。
    '''
    jobs = BUILD_JOBS

    '''
    Report the CPU concurrency and available host memory before compilation.
    編譯前顯示可用 CPU 核心數與系統可用記憶體。
    '''
    cpu_slots = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 2)
    print(f'Build jobs: {jobs} | Available CPU slots: {cpu_slots}', flush=True)

    '''
    Warn when jobs are higher than CPU slots; excess parallelism may be slower.
    當平行編譯數高於 CPU 可用核心數時提醒，過高不一定比較快。
    '''
    if jobs > cpu_slots:

        '''
        Encourage users to compare six and eight without hiding the setting.
        提醒使用者可以比較 6 與 8 的編譯效果。
        '''
        print('Notice: Build jobs exceed CPU slots; try BUILD_JOBS=6 if slower or memory-limited.', flush=True)

    '''
    Configure CUDA for T4 and disable unsupported CUDA VMM.
    針對 T4 設定 CUDA，並停用不相容的 VMM。
    '''
    command = [
        'cmake', '-S', str(PROJECT), '-B', str(PROJECT / 'build'),
        '-DGGML_CUDA=ON', '-DGGML_CUDA_NO_VMM=ON',
        '-DCMAKE_BUILD_TYPE=Release', '-DCMAKE_CUDA_ARCHITECTURES=75',
        f'-DCMAKE_CUDA_COMPILER={NVCC}',
    ]

    '''
    Prefer Ninja for new builds; leave existing Makefiles unchanged.
    新建置優先使用 Ninja，但不更改既有 Makefile 專案。
    '''
    if shutil.which('ninja') and not (PROJECT / 'build/CMakeCache.txt').exists():

        '''
        Select Ninja only before the initial CMake configuration.
        僅在首次 CMake 設定前選用 Ninja。
        '''
        command.extend(['-G', 'Ninja'])

    '''
    Reuse ccache when installed; it speeds up later rebuilds.
    若已有 ccache，則重複編譯時沿用其編譯快取。
    '''
    if shutil.which('ccache'):

        '''
        Enable the cache for both C++ and CUDA compilation.
        讓 C++ 與 CUDA 編譯都使用快取。
        '''
        command.extend([
            '-DCMAKE_CXX_COMPILER_LAUNCHER=ccache',
            '-DCMAKE_CUDA_COMPILER_LAUNCHER=ccache',
        ])

    '''
    Configure the native CUDA executable without touching model weights.
    建立原生 CUDA 編譯設定，不影響模型權重。
    '''
    subprocess.run(command, check=True)

    '''
    Compile the persistent server with bounded parallelism.
    使用受控平行度編譯常駐服務。
    '''
    print(f'Building yue-server with {jobs} parallel jobs...', flush=True)

    '''
    Reuse completed objects when the build is resumed.
    中斷後重新執行時沿用已完成的編譯物件。
    '''
    subprocess.run(
        ['cmake', '--build', str(PROJECT / 'build'), '--target',
         'yue-server', '--parallel', str(jobs)],
        check=True,
    )

    '''
    Record the revision only after a successful build.
    只有編譯成功後才記錄完成的原始碼版本。
    '''
    BUILD_STAMP.write_text(CURRENT_REVISION + '\n', encoding='utf-8')
else:

    '''
    Reuse the existing executable without recompilation.
    直接沿用已有的執行檔，避免重複編譯。
    '''
    print('Reusing compiled yue-server; CUDA build skipped.', flush=True)

'''
Verify the compiled executable and identify the source revision.
確認執行檔存在，並記錄原始碼版本。
'''
assert SERVER.is_file(), 'yue-server 編譯失敗。'
revision = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True,
).strip()
print(f'YuE2 server ready (commit {revision}).')

## 2. 下載模型


In [ ]:
'''
Download the original-precision backbone and VAE only once.
只下載一次 BF16 原始精度主模型與 F32 VAE。
'''
from huggingface_hub import hf_hub_download

'''
Share the same model files between the two GPU servers.
讓兩個 GPU 服務共用同一份磁碟模型檔案。
'''
MODEL_DIR = ROOT / 'yue2_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPO = 'Serveurperso/YuE2-GGUF'
BACKBONE = Path(hf_hub_download(
    repo_id=REPO, filename='YuE2-3B-BF16.gguf', local_dir=str(MODEL_DIR),
))
VAE = Path(hf_hub_download(
    repo_id=REPO, filename='YuE2-Vae-F32.gguf', local_dir=str(MODEL_DIR),
))

'''
Stop if downloaded weights appear incomplete.
如果權重檔案明顯不完整就停止。
'''
assert BACKBONE.stat().st_size > 6 * 1024**3
assert VAE.stat().st_size > 400 * 1024**2
print('Models:', BACKBONE.name, VAE.name)

## 3. 啟動單 GPU 常駐服務（GPU 0 或 1）


In [ ]:
'''
Start one resident model service on the selected GPU.
只在指定 GPU 啟動一個模型常駐服務。
'''
import requests
import time

'''
Assign separate localhost ports and preserve process handles.
指定獨立本機連接埠並保留程序控制項。
'''
PORTS = {0: 8091, 1: 8092}

'''
Choose a single GPU (0 or 1) before starting the resident model.
在載入常駐模型前，指定一張 GPU（0 或 1）。
'''
GPU_ID = 0

'''
Reuse existing processes if this cell runs again.
重新執行時沿用目前已啟動的程序。
'''
SERVER_PROCS = globals().get('SERVER_PROCS', {})

'''
Start only the selected GPU service and preserve an existing healthy instance.
只啟動選取的 GPU 服務，健康的既有程序則直接沿用。
'''
assert GPU_ID in PORTS, "GPU_ID 必須為 0 或 1。"

'''
Keep the model on one GPU instead of loading two independent copies.
只在單張 GPU 載入模型，不額外複製第二份。
'''
for gpu_id in (GPU_ID,):

    '''
    Resolve the selected GPU service port.
    取得指定 GPU 的服務連接埠。
    '''
    port = PORTS[gpu_id]

    '''
    Inspect an existing service before starting another process.
    啟動新程序前先檢查原有服務。
    '''
    address = f'http://127.0.0.1:{port}'
    process = SERVER_PROCS.get(gpu_id)
    try:

        '''
        Check both service health and the model it is serving.
        同時確認服務健康狀態及載入的模型路徑。
        '''
        healthy = requests.get(address + '/health', timeout=2).ok
        props = requests.get(address + '/props', timeout=2).json() if healthy else {}
    except requests.RequestException:

        '''
        An offline or initializing service is not reusable yet.
        尚未啟動的服務不能直接沿用。
        '''
        healthy, props = False, {}

    '''
    Reuse only the matching backbone to avoid an incorrect server.
    只有模型路徑相符才沿用，避免連到其他服務。
    '''
    if healthy:
        assert Path(props.get('model', '')).name == BACKBONE.name, f'Port {port} 已由其他模型占用。'
        print(f'GPU {gpu_id}: reusing resident server.')
        continue

    '''
    Avoid launching a second server while the first is still loading.
    原服務仍在啟動時，避免再啟動第二份模型。
    '''
    if process is not None and process.poll() is None:
        raise RuntimeError(f'GPU {gpu_id} 原服務尚未就緒，請查看 yue2_gpu{gpu_id}.log。')

    '''
    Keep AR, NAR and VAE weights resident after the first request.
    首次生成後將 AR、NAR 與 VAE 權重保留在 GPU。
    '''
    command = [
        str(SERVER), '--model', str(BACKBONE), '--vae', str(VAE),
        '--device', f'CUDA{gpu_id}', '--host', '127.0.0.1',
        '--port', str(port), '--keep-loaded', '--max-seq', '8192',
        '--max-batch', '1', '--vae-core', '256',
    ]

    '''
    Store errors in a per-GPU log for diagnostics.
    將錯誤紀錄分別儲存到各張 GPU 的 Log。
    '''
    log_path = ROOT / f'yue2_gpu{gpu_id}.log'
    with log_path.open('a', encoding='utf-8') as log:

        '''
        Start the server independently of the notebook cell.
        讓服務在此 Cell 結束後繼續運作。
        '''
        process = subprocess.Popen(
            command, cwd=PROJECT, stdout=log,
            stderr=subprocess.STDOUT, start_new_session=True,
        )
    SERVER_PROCS[gpu_id] = process

    '''
    Poll readiness while checking for an early process exit.
    等待服務啟動，並檢查是否提早退出。
    '''
    for _ in range(120):
        if process.poll() is not None:
            raise RuntimeError(log_path.read_text(errors='replace')[-3000:])
        try:
            if requests.get(address + '/health', timeout=2).ok:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    else:
        raise TimeoutError(f'GPU {gpu_id}: {log_path.read_text(errors="replace")[-2000:]}')
    print(f'GPU {gpu_id}: server ready (port {port}).')

'''
Clarify when server weights actually enter VRAM.
說明模型在首次生成時才會真正載入 GPU。
'''
print('BF16 weights load on first generation and remain resident afterward.')

## 4. 生成設定


In [ ]:
'''
Describe the musical style and vocal performance.
設定曲風與演唱方式。
'''
STYLE = '''
Mandarin pop, warm female vocal, acoustic guitar, soft piano, ballad, 90 BPM
'''.strip()

'''
Write the lyrics with section labels.
使用段落標記設定歌詞。
'''
LYRICS = '''
[Verse]
微風吹過熟悉的街道
晚霞落在回家的轉角

[Chorus]
讓心裡的光慢慢閃耀
明天還有新的美好
'''.strip()

'''
Control duration, sampling, and score planning.
設定秒數、生成步數與樂譜規劃方式。
'''
DURATION = 30
STEPS = 32

'''
Available choices: full, melody, off.
可選值：full、melody、off。
'''

COT = 'full'

'''
Set separate language-model and audio-synthesis seeds; -1 means random. Use 42 for reproducible seeds.
分別設定語言模型與音訊合成種子；-1 代表隨機。 設為 42 可以固定種子並重現結果。
'''
LM_SEED, SEED = -1, -1

'''
The selected GPU_ID comes from the resident-service cell.
GPU_ID 由前面的常駐服務 Cell 指定；每次僅生成一首歌曲。
'''

'''
Set a filename that will never overwrite previous results.
設定輸出檔名，產生新版本時不覆蓋舊檔。
'''
OUTPUT_NAME = 'yue2_bf16.mp3'

'''
Supply an edited ABC file to reuse its musical plan; leave empty normally.
若要依修改後的樂譜重製歌曲，填入 ABC 路徑；一般生成留空。
'''
ABC_PATH = ''

'''
Preview the score and convert to MIDI when tools are available.
工具可用時預覽五線譜並轉換 MIDI。
'''
SHOW_SCORE = True
EXPORT_MIDI = True

## 5. 生成音樂（可重複執行）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import json
import time
import requests

'''
Read and write files under Kaggle working.
讀寫 Kaggle 工作目錄中的檔案。
'''
from pathlib import Path
from email import policy
from email.parser import BytesParser

'''
Validate the generation settings before requesting GPU work.
執行 GPU 推論前先驗證生成設定。
'''
assert GPU_ID in PORTS, 'GPU_ID 只能設定為 0 或 1。'
assert COT in ('full', 'melody', 'off') and DURATION > 0 and STEPS > 0

'''
Reuse a single output folder between generations.
各次生成皆沿用相同輸出資料夾。
'''
OUTPUT_DIR = ROOT / 'yue2_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

'''
Optionally load a user-edited ABC score to guide music generation.
視需要讀入使用者修改的 ABC 樂譜作為生成條件。
'''
score_text = Path(ABC_PATH).read_text(encoding='utf-8').strip() if ABC_PATH else ''
assert not score_text or COT != 'off', '提供 ABC_PATH 時，COT 應設定 full 或 melody。'

'''
Build a request matching the official yue2.cpp API.
建立符合 yue2.cpp 官方格式的生成請求。
'''
request_data = {
    'style': STYLE, 'lyrics': LYRICS, 'abc': score_text,
    'cot': COT, 'duration': DURATION, 'steps': STEPS,
    'lm_seed': LM_SEED, 'seed': SEED, 'lm_batch_size': 1,
    'synth_batch_size': 1, 'output_format': 'mp3', 'mp3_bitrate': 192,
}

'''
Generate on one designated server without starting a new model process.
使用指定服務生成歌曲，不重新啟動模型程序。
'''
def generate_on_gpu(gpu_id):

    '''
    Access this GPU's local API endpoint.
    取得這張 GPU 對應的本機 API。
    '''
    address = f'http://127.0.0.1:{PORTS[gpu_id]}'
    submitted = requests.post(address + '/synth', json=request_data, timeout=60)
    submitted.raise_for_status()
    job_id = submitted.json()['id']
    print(f'GPU {gpu_id}: job {job_id} started', flush=True)

    '''
    Poll the job state without repeating log messages each second.
    定期查詢生成進度，避免每秒重複輸出相同訊息。
    '''
    deadline = time.monotonic() + 3600
    last_report = time.monotonic()
    while time.monotonic() < deadline:

        '''
        Check the current status and handle failures explicitly.
        檢查工作狀態，並明確回報失敗原因。
        '''
        status_response = requests.get(address + '/job', params={'id': job_id}, timeout=30)
        status_response.raise_for_status()
        status = status_response.json()
        if status.get('status') == 'done':
            break
        if status.get('status') in ('failed', 'cancelled'):
            raise RuntimeError(f'GPU {gpu_id}: {status}')
        if time.monotonic() - last_report >= 30:
            print(f'GPU {gpu_id}: still generating ({status.get("status")})', flush=True)
            last_report = time.monotonic()
        time.sleep(5)
    else:
        raise TimeoutError(f'GPU {gpu_id}: job {job_id} exceeded 1 hour.')

    '''
    Download the official multipart result with replay metadata and audio.
    下載包含重播參數與音訊的官方 multipart 結果。
    '''
    response = requests.get(
        address + '/job', params={'id': job_id, 'result': 1}, timeout=180,
    )
    response.raise_for_status()
    mime = (
        b'MIME-Version: 1.0\r\nContent-Type: '
        + response.headers['Content-Type'].encode()
        + b'\r\n\r\n' + response.content
    )
    message = BytesParser(policy=policy.default).parsebytes(mime)
    assert message.is_multipart(), f'GPU {gpu_id}: unexpected result format.'
    parts = list(message.iter_parts())

    '''
    Preserve the audio and the reproducible score metadata separately.
    分別保留 MP3 音訊與可重現的樂譜資訊。
    '''
    audio = next(
        (part.get_payload(decode=True) for part in parts
         if part.get_content_type() in ('audio/mpeg', 'audio/mp3')),
        None,
    )
    assert audio and len(audio) > 4096, f'GPU {gpu_id}: empty MP3 response.'
    replay = next(
        (json.loads(part.get_payload(decode=True)) for part in parts
         if part.get_content_type() == 'application/json'),
        None,
    )

    '''
    Allocate a unique filename for this GPU's output.
    為這張 GPU 的歌曲建立不重複的檔名。
    '''
    stem = Path(OUTPUT_NAME).stem or 'yue2'
    index = 1
    while (OUTPUT_DIR / f'{stem}_gpu{gpu_id}_{index:03d}.mp3').exists():
        index += 1
    output = OUTPUT_DIR / f'{stem}_gpu{gpu_id}_{index:03d}.mp3'
    output.write_bytes(audio)

    '''
    Save the exact replay request and generated ABC when available.
    若有產生樂譜，儲存完整重播參數及 ABC。
    '''
    if replay is not None:
        output.with_suffix('.json').write_text(
            json.dumps(replay, ensure_ascii=False, indent=2), encoding='utf-8',
        )
        if replay.get('abc'):
            output.with_suffix('.abc').write_text(replay['abc'], encoding='utf-8')
    print(f'GPU {gpu_id}: saved {output.name}', flush=True)
    return output

'''
Run one song on one GPU and keep its output for the preview cell.
只在一張 GPU 生成一首歌曲，保留結果供試聽 Cell 使用。
'''
OUTPUTS = [generate_on_gpu(GPU_ID)]

## 6. 試聽、樂譜與下載


In [ ]:
'''
Preview audio and render optional scores without loading GPU models.
試聽歌曲並顯示選用樂譜，不重新載入 GPU 模型。
'''
import shutil
import subprocess
from IPython.display import Audio, FileLink, SVG, display

'''
Install the lightweight score tools only when requested and absent.
只有需要樂譜功能且工具不存在時才安裝輕量套件。
'''
tools = [name for name, enabled in [('abcm2ps', SHOW_SCORE), ('abc2midi', EXPORT_MIDI)]
         if enabled and shutil.which(name) is None]
if tools and not globals().get('SCORE_TOOLS_ATTEMPTED', False):

    '''
    Avoid retrying apt during every song preview.
    避免每次試聽歌曲時重新執行 apt 安裝。
    '''
    SCORE_TOOLS_ATTEMPTED = True

    '''
    Install the Debian packages if the Kaggle runtime permits apt.
    若 Kaggle 環境允許 apt，才安裝樂譜排版與 MIDI 工具。
    '''
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'abcm2ps', 'abcmidi'], check=False)

'''
Show every generated audio file and its available outputs.
顯示每個音樂檔案及可下載的相關成果。
'''
for output in OUTPUTS:

    '''
    Play the completed song directly inside Kaggle.
    在 Kaggle 直接播放生成的歌曲。
    '''
    print(output.name)
    display(Audio(filename=str(output)))
    display(FileLink(str(output)))

    '''
    Display reproducibility files and the editable ABC score.
    顯示可重現設定與可編輯的 ABC 樂譜。
    '''
    replay_path = output.with_suffix('.json')
    score_path = output.with_suffix('.abc')
    if replay_path.is_file():
        display(FileLink(str(replay_path)))
    if not score_path.is_file():
        print('No ABC score returned (for example, COT=off).')
        continue
    display(FileLink(str(score_path)))

    '''
    Use abcm2ps to export a standard SVG staff notation.
    使用 abcm2ps 將 ABC 排版成標準五線譜 SVG。
    '''
    if SHOW_SCORE and shutil.which('abcm2ps'):
        prefix = f'{score_path.stem}_sheet'
        rendered = subprocess.run(
            ['abcm2ps', '-g', '-S', '-O', prefix, score_path.name],
            cwd=OUTPUT_DIR, capture_output=True, text=True,
        )
        if rendered.returncode:
            print('Score renderer:', rendered.stderr[-400:])
        else:
            for score_svg in sorted(OUTPUT_DIR.glob(f'{prefix}*.svg')):
                display(SVG(filename=str(score_svg)))
                display(FileLink(str(score_svg)))

    '''
    Convert ABC to MIDI when the notation supports conversion.
    樂譜格式相容時才將 ABC 轉成 MIDI。
    '''
    if EXPORT_MIDI and shutil.which('abc2midi'):
        midi_path = score_path.with_suffix('.mid')
        converted = subprocess.run(
            ['abc2midi', score_path.name, '-o', midi_path.name],
            cwd=OUTPUT_DIR, capture_output=True, text=True,
        )
        if converted.returncode == 0 and midi_path.is_file():
            display(FileLink(str(midi_path)))
        else:
            print('MIDI conversion unavailable for this ABC:', converted.stderr[-300:])

## 7. ngrok 與遠端生成 API 啟動

In [ ]:
'''
Dependencies are available in the Python standard library.
只使用 Python 標準函式庫處理遠端服務與檔案。
'''
import zlib

import queue
import copy

'''
Use secure randomness when the API requests seed=-1.
API 指定 seed=-1 時，以安全的隨機來源產生實際種子。
'''
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

'''
Expose only the inference bridge, never the Jupyter or ComfyUI management interfaces.
ngrok 僅轉送推論橋接 API，不公開 Jupyter 或 ComfyUI 管理介面。
'''
REMOTE_PORT = 8765
REMOTE_SERVICE_ID = '06_YuE2_音樂生成_Kaggle'
REMOTE_KIND = 'music'
REMOTE_FIELDS = ['task', 'style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot', 'abc']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\x9dXkS\xdc\xd6\x19\xfe\xee_q\x9a|\x90v\x0c\x02L\xeb&L73\x18\xd3\x94il\x18\xc0\x99\xc98\x9e\x1d\xadt\x96U\xac\x95T\x9d#\xc3\x96a\x06\x88\t8\x80a2\x06_ \xc6vm\x82/\xe0\xd8Nj\x82\xb1\xfdc\x8a\xb4\xbb\x9f\xfa\x17\xfa\xbe\xe7H\xbbZ\xb0\xd3\xb4x\xc0\xba\x9c\xf7\xfe\xbc7}H\x86\x82|\xc9\xe2\xc4u(a\xae3B\xb8Kx\x11\xae\xa9M\rNM\xe2Sf\x99\xd4\xe1\xe4\xd3\x81s\xda\xb1\x0fI\xf8\xe3l\xb8\xbaS\xdb\\\x8d\xb6\x17\xa2\xb5\xe7\x95[\x97+\xd76\xa2\xb9\xe5\x83W\xaf\xc2\xf9\x9b\xb5\xc9\xa9\xea\xec\x8bha6\xdc\xb9\x05\xaf\xc2\xdd\xdd\xda\x0f\xcbHJ\xa2\xf5Ex\xff\xaf\xc9\xe9cV\xc9s}N\xbe\x02q\xc95\xb7J\xf4\x180\xef6\x0c\xca\x98P@\xb7}\xaa\x9b\xe5V?p\x1c\x0b\xf4\xb2]C\xb7\x89\xe5\x14\xa8O\x1d\x03\x15\xf4/Yp\x1au\xaaM\xfe\xa3\xf2\xf2\x16\x08\xae\xac\xed\x84W\x16\xa3\x85\xa9\xca\xdeBmj\xfe\xe0\xf5:(\x11]\xdd\xaa>^="\xdf\xa7\x7f\x0b(\xe3\x0c\xe5\x0e\x82(\xa2;&\x19\xf5-NI\xc1\xb2)#\x81cR\x9f\xfcU\x1f\x19\xb1)\x19u\xfd\x8b\xa0\x06J\xab\xeeL\x86O\x1f\'/\xc2\x97\x0fP\xcc\xdaNm\xe1\xf2\xc1\xee6\xca{t-\xba;\x87\x92\n\xbe["\x9e\xce\x8b\xb6\x95\'\xb1\xd8\x01\xb8\x95/hI\xb7\xec\xe4\xb1\xe7\xda\x96QN\xbd\xd0<\xdd\x07\x1b\x93\xf7\xa7\xca\x9c\xb2\x01\xf1\x08\x15\xfe\\\xb7-S\x07U\xd1U#\xd4\xa1\xbe\xce-\xd7\x01\xafp\x0ej2\x92\xa7\x05\xd7\xa7\x89\x91\xe8@\x0c\x02Z!\x82\xb8\xf1\xb2zwA\xc6Ex\x07\xbc\x16\xce\xcc\xd5\x1e^\xafn_\x91\xf1\xacnmC\x0c\xd1\n\x9d\x81P\x11\xff\\\xdfi\x88\x00\x19\xe8\x1f\x1c\x1ej!J\xfc$\\zT\xfd\xfa\xb5<_\x99\xde#\xed$\x9a[%\x1d@\xaa$\xb4=\xfd\xc3H\xa8*\x85\xc0\xb6\x15 -Q\xdb5\xcbx\xe5\x16\nJF\xf8\xfe\xf4\xb9\xc1\xee\xe1\xbe\xfe\xb3\xe4\x13`\x81\x0f\x86\x86{\x07\x86\xf0N\x86(`\x80\n\xc2\xc0\x16p\xbb\x1bp/\xe0\xa4\xe0\xda\x18\xa4<\xe5\xa3\x94:)O\x08\\\x84\xcb\x97\xa3\'w\xa5=\x95[\xdfD\xcf\xdfV\xaemU\xd6v\xc3\xe5\x85\xea\xfen8\xbbW}1\x1b\xad\xde\x0c\xef\xbfA;\xfb\xcf\r\x0f\x9c\x1b\xce\x9d\xee\x1b$Y2\xd8\x0f*\xb7\x11\xa5\x1c\xd0\x139)\x8c)\xa9#Z\xe9\xa2i\xf9*\x04\tr\x83e\x87\xfd\x80\xb6\x10:f1\x9es/\x8a\xdb\x0cj\xdd\xef\xa12\xbam\x97\x01\xc0\x881\x02V\xf8\xad\xd4\xb40\xb7\xbaO\xf5\x10f`\x9c \xebF\x02\xc83R\n\x98e\xa4\xec\x10\x80\xdb\\\xad\xadOV7\xa7\x10y3\x0074\xa3:9s\xf0v\'\xba\xf6\x0b N0\x8a\xb6\xa6\xab\xdb\xeb\x88\xc5\xe9=irt\xef\xfb\x83W\xffD\xd3\x84\x90\x1c\xa7c\x1cLC\x04\xaa@\x91\x1b\xe8\x1e\xfeKF\xc34\x13\xafT\xc8+\xd7\x04\xf7f\x95\x80\x17Z?R2\x1a\xe3\xbe\xe5\xa9\x19b\x15HB@\xa8\rqP\xea\xa1u\\NR\xec]_\x04\xfbwY\x19Y\x08p\xb4\xb4|\xf0f\xadA\x1f\xdd\x9c\xfe\xf7\xfe\x02\x1e\x8af\xafH\xd4\x10\x84\x85@\x8d\xc4\x85\x80\x0e\x18~*\xb0l\xf4Y\x8cbR\xd2\xb9QD,#\xea\x81\xbdeXP\x130D\x9a\xe1y\xa4{\xa0OD\xfd\xd5^\xe5\xf1|\xe5\xc9f\xb8<\xd7x\x19\xee\xdc\x88V\x7f\x89\xee\xec\x87\xfbK\xf5\xa2U}<\x1f=\x9bF\x0f\xc5"r\x90T:\xf8h\xfc\x18\x81\x1f\x85\xf1\xb2M\x95.@\xe2\x17\x9f\xf5\x82-v\xd9\xb7\x0c\x06\x0f>\xfbb\xb0\xaf\x07s@\xcf\x1bp\xdbp@\x8b$4\\\x0e\x8f\xc1H8b\x062\x94\xf0 \x81x\x0b\xb2\xa6\x1e\x13\xac\x01\xe41\x95]\xca1JM\xe4\x7f&7\xd4\xdb{\x1a\xcf\xc9\x07\xf1\x1d\x9c\xc8\xa3\x13r\xcc\xfa;*\xd6\x11S\xb2\xb2\xc3\x8b\x87_A\x08\x04psP\n\xc0u\xf0L)y\x9d"\xfd\xbc\xce\\\xde\xe2\xa0\x978\xf9\xf1\x89\x96c\x13\xe8\xf0O%\xee\xc0\xb9\x8e\xe8\x08&\x14\xff\x11GG\xacb\xbd\x85L\x1b\xb5x\x11\xb8\x12\xc6u_\x94\x15\x9d8t\x94\x94\\\x93\xda\xc4\xf3]\xac\xdf\x18\x04\tR\xd9\x08d\xe9\x8d1)\xba\x06\x00\xe0`w\xb16\xbb\x18\xad\xfe\x18\xael\x84\xf3+\xd1\xd6\xdd\xf0\xf6|ek>\xdc[\xc2x\x98\xb4\x90d\x01\xcd\xb9Nn\xc4\x0bT\xf8\xcdYf\xa6K\x98\x9cj\x16\x16\xc3\xe2\xa4\xb0\xb8G\x00\x0c\x08uL\xcf\xb5\x1c\xae\xc5g\xc3\xa5\xd5\xf0\xcd\xf5\xda\xe4\xcdp\xffgQ\xf6\xc2\x1f\x17\x01~X\xad\xd7\x9fD\x0f7\x90\x08\xc5\xe2i\xdd4}d\x9c%\x05\xa5\xc8\xb9\xd7\xd5\xd6\xd6q\xe2\x8fZ;\xfc\xeb\xe8\x1a\x17\xc5\xef\xbcT\xe5\xc2\x84"(\x98\xe8\xa1\xe8\xa5l\xbd\xadh\x9e\xcb\xb8\x9a\xf0:N\x946\x11#\xf0>6\xbel\x1ao-\xa2\xff\x81W\xb3\'\xdb3\xcd\x0c5_\xb7\x18\xc5\x00\xe6\xc0\xe3<`\xaa<\xf0\x95\x9b\x07\xf9 \xaeq\x12\xd9\xaa\x99\xf3\x8ae*\x17\xc4\x19\xcf\x07\x07\xa8\x05,\xd3d\\*<\xd1\x85\x94d\\\x92O\xc8(\x02\xbeZH\xc1\x0eX1\xa9[\xd2g\x03.$%f\x1a\x92\xa0tZ\x8f\xbeO=\xaas\xd9\x96G i\x19\xd3G\xa0cR\xdd(\x02P\x0c\xd71\xeb\x8e\xc7\xf0oD\x1b\x0f\xaa\x0f\xefI\x04\xd4&\x9f\x87{\x9b\x80\x80\xda\xd4\xdbpf1z\xbaT\xf9\xe1;\x80B\xf5\xfe\xac,\xcaq\x81\xde\xfa6\x9az\x9a\xc4\xc4\x84"e[\x80\xc8\xac\xf0\x95Vr\xa1\xee\xb8\x8ee@q:N:O\xb6\xb7\x8bc\xb6\x0e.\x05\xe5\xb0c\x1e=)\x8e\x8c\x16\xa1\xb9\x1fe\xf2\xa7\xba\x08\t/\xa9}O\x91\x1a\x17\x85\x0f\x8c\xc0\xc7JOd\x14D{*\xc2\x1f`U\x80f\x1d\xf8h\xfd\x98\x07=\xdc\xe2vYK\xb1\x88\x1e\xdd\x03\xeb\xe3)a~2\x9a\x99\x17\xe0\xdf\x8cn\\\xad\xdc\xdb\x0b\xd7n\x87w\x9e\x85\xf7\x9fE+\xd7\xc3\xab\x1b\xe1\xda\x9d\xc4b\x01\x03!\x0c\x0cb\x1et4\x9aF\xd7\x08m\x06\x17\xc4\x08\xa2\x08\xedH/\xb1\xec8\x82\xa0+\x06\xc9D\x03^\x9d1\xbc\xde\xc1\xfb}@k\x1cE\xb0\x1d\xa2\x91\x90\xab\x9f\x83.!O\x08\xed\x14y\r\xcd=\x0b\xcd\xc0\x84r\xa24\\\x8b?y\xe8<\x17\xff+\xb1\x1c\x1b\xc0\xc7\x02\xa6\x8a\xa1\xc3\xf0g\xe3M\xa6\x99\x9b\xd0\x9f\x0c\x06\x0e\x1a\xdb\xeb\xfb\xae\x7f\x04\xfa\xe3\x92\xe9\x84\xd2\xa4\xf2\x11(\xb46\xc1\xe8\x93,\xe9lo\x16\xf5\xee\xc4\x82)\x0b\xf2%\xe9\xdc\x90\x1a\xeax\xca\xa0\x0f\xe4\xf5\x07\x99\x89\xcc\xa1lKs\xfe\r\xf8\xc5\x1f\xf1\x82\xd9\x94z\xea\x1f\xe4Sl\xca\r%\xa5/\x86e\xd4\xdf\xed\x8b\xa62@\xc7\x0c\xe82P\xbe:\x08\xa4\xb7\xaf)\xf5\x1ap\xda\x1du\xc4\xe0\xd2\xd4qK\x81\xcd-\xc0\x1a\x16\x02\x06\xd7\xa2.`Q\xb0\xf52T\x03\xae\x8b.\x8a9\xa2\x07\xa6\xe5&\xb5\xe0`w\xbe\xba\x7f%\\\x98\t\x97\x1fc\xed\xffn;\\\xfa:Z\xd9\xad\xce\xcd\xd66^@\xc2\xe3\xc6 \xdatJB\xe5\xe7\xe5\xe8\xf6z\x92\x15\xefK\x85\xba\xe9\xbf-\'\x00HRs\xec~\xa9\x04\xe9\xf8\xa8]6\xd4L\x93\xb4\xf7%G\t\xc8@\x93\x86\xf4\xbcr\xa6\xefLo\xeb\xe7\xd4g\xd0\xf0\x81\xb7\xd6\xfe\xa5\xff\xa5\xd3\xe3:\x1c\x8aG\xebp\xd9\xa3\xd0\x84\xeb\xc7\x8f7$\x14\xa1\xfa\x00\xd5y%}V\xb9\xa0\x89\x99\x8c\xa6b\x7f\x1c\x84 O\xfcU\xd2\x1c\x0cI\x98R?\xae\xcb\xa0aj{P\xe5\xa6\x91\x95\xffi\xd0gu\xf0CF\xae\x1by<\xa7\xa2Y\x92C<\xe3\xc5\x8c4\x8b\xe5\xea\x81Q3\x00\xe3C\xa0\n\x1c\xa8\x81\xf5\xb5\x11\x81!\'\x0fM\x9a\x8cdXFl\x98\x92\xd5:ON\xfd\x9cx\xa3f\x1a\xad\x07\xc8q\xde\x90\x9b bH\xa0\t\xef\x00e\xbek\x06\x86\x95\x87\xda+\x87\xe7:\xe2\x18\xc5`s\x9aT`h?s\xdf\x84s\x0f\x0e\xde~_Y\xb9I\xce\x0ct\x12\t4@\\\xb8\xf4\x140XYz\x833\x80\x18\x9da\x19\x80W\xf5)@H\xcd\xc2t3\x96\x82\x17\x0e\xfc\x1c\x11\x07*\x9711T\x93b\x80d.\xa3\xb9\xc2J\xacZ\xc2\xa6FvC\xa5\xa9\xd3\xc6\xa1\xcaq\x88\xb1\x1a\x978!\xaf\xad\xe4\xd1\x11,s\xc9]\xa7\x92\xc9\xb4\xd4\x99\x9c\x85"\x9a\xc6g\x1c\x9e\x86\x83l\xea\xa8\xe2.\x03k\xd3\xef\xdb?>y4F\xb4\xe4\xf1\xb2pE\x1d9J\x0cv\x91\xbeG,\xc6"\xaf\xa1\xa9\xec\xd7\x8d\xff\x7f\xad\xc7\xee\xa0{\xd8:\xc5\x94\xdc\x86\xf2\x94\xf7\x18\x9dL~6Nz\x1c\xd7\xc1\xc0\xb1\xa0\x0c\x88\x8d\xdd\xd1!\x1bQ\x87\xd4D(\x07\xe0\x04\r\xb0\x19\xa5F@\x0c\xbc\x98F\xe5\xc2 \x07R\x98B\xe4\n\x1f./&P\x80Q\xbd\x94\xecM\xf1\x06x\xb6\xfbL/\xaeG\xf0\x02\x04\x8a-Qz\xd1rL:\x06\x87;R\xc3\x86\x9a\xda,\xdb  \xe3H7\x81#m\x12\x96\xdc\xb8\xa0\xebj\xef4\'4\x11uM\xec\x92\x90\x13\x8d\xaa.Y\x1fOx\xc7\x1bp\x96\xfc\xef\xdcS\xf4\x9a\xf8\xe2\x91\x93\x89/\xa1\x93xyH\x8f\x13\x90\x8e\xe9\x06O\xf0\x91\xecb\x08\xb7d@\x97\x9b\xech\x116p\xfd\x12\xb4j\x1dr3\xf1yu\xfeA\xb4~\xa5rML\x7f"\xcf`\xfe\t/?\x0f\xb7o\x84;\x0b\xd1\xcaO\xe9N\x10.}\x8b\xac\x12\xc7\x03jb\xa9\x10P\\4\x11\x0b\r\x87$\x16@\xf3\xc9\xb1\x00\x9a\xd3\x98\xaah\x12>\xb1Y\xbc\t\xcabpF4\x9bA\xc9c\xaad\r[\xbb\xc3`~\xcb\xe9\xcc\xb0\xac\xec\x9fu\xe8\xa4-\xc2\xd7\x0e\xcf\x9e\xc8\xe0\xeb\xe6\xad\xb8\x81\xcc\xa61Br\x93\xc3\x0b.\x85\x87\xc6\x93w\xea*\xce\xa5U\x95L\xce\x0b\x06\x17\x8e\x8a\xce\xfc\xca\\\xcf Z&\x19\x8f\xe5`.L\xbcc\xd0\xf0)\x0f|\'\xd6F|V\t\x9c\xc6\xb7\xbfx\xebC\xc6\x18\xde\x8b0`\x10\x8b\xb3\xc6\xc7\x16_\xe0\xc1\xf3\xe9%\x0b\x96>\x1c\xc4\xc4\xc2\xbd\xf4(\\\xdf:\xd8\x9d\xacg\x96\xfc"\xb8;Y\xffL\x883\xaf(\xc2\xb2\x9d\x1f\xbcY\xab>\xdc\xacN\xbd&=\xc0#^\x16\x1b\xdf`\x86\x00\xd5\xe7\x0fo\x7f\xf2kS\xe6\xc2\x7f\x00a\xba\xcf\xa5').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['music']
REMOTE_MAX_IMAGES = 0

'''
Keep job directories isolated between different notebook services.
各 Notebook 使用獨立工作資料夾，避免不同服務互相干擾。
'''
REMOTE_ROOT = Path('/kaggle/working/remote_api_jobs') / REMOTE_SERVICE_ID
REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('REMOTE_ENGINE_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=32))
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(2))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = 24 * 3600
REMOTE_RESULT_BYTES = 8 * 1024**3
REMOTE_MIN_FREE_BYTES = 2 * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = 2 * 3600

'''
Report only checks that can be verified without loading another model.
只檢查不會額外載入模型的服務就緒條件。
'''
def remote_backend_ready():
    if REMOTE_KIND == 'image_multi':
        return bool(globals().get('QWEN21_STATE', {}).get('pipeline'))
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        return globals().get('pipe') is not None
    if REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        gpu = globals().get('GPU_ID', 0)
        process = globals().get('SERVER_PROCS', {}).get(gpu)
        return bool(process and process.poll() is None)
    base = globals().get('BASE')
    if not base and REMOTE_IS_LTX:
        base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
    if not isinstance(base, str):
        return False
    try:
        with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=3) as response:
            return response.status == 200
    except (OSError, ValueError):
        return False

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'backend_ready': remote_backend_ready(), 'stalled': remote_stalled(),
                                 'running': running, 'waiting': REMOTE_QUEUE.qsize(),
                                 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return

        '''
        Reuse the same job ID for retry attempts after an uncertain response.
        網路回應不明時，重試會沿用同一個工作 ID，避免重複生成。
        '''
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        if not remote_backend_ready():
            self.send_json(503, {'error': 'Backend is not ready; execute model initialization cells first'}, 20)
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):

                '''
                Ignore missing seeds and validate the supported 32-bit range.
                未提供的種子沿用 Notebook 設定，並檢查 32-bit 範圍。
                '''
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')

                    '''
                    Translate -1 into an explicit random value before inference.
                    推論前將 -1 轉成明確的隨機數值。
                    '''
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)

            '''
            Save supplied seeds for reproducibility in the job status API.
            在工作狀態 API 中記錄請求所使用的實際種子。
            '''
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        '''
                        Point R2V to this job's uploaded reference images only.
                        讓 R2V 僅讀取這筆工作的參考圖片資料夾。
                        '''
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:

                    '''
                    LTX-2.5 uses a shared ACTIVE_JOB configuration.
                    LTX-2.5 使用共用 ACTIVE_JOB 參數。
                    '''
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'

                    '''
                    Avoid carrying an ABC score from a previous local Notebook run.
                    避免沿用前次 Notebook 手動生成時的 ABC 樂譜。
                    '''
                    namespace['ABC_PATH'] = ''
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'

            '''
            Clear old output metadata so a failed generation cannot reuse another job's file.
            清除前次輸出狀態，避免本次失敗時誤取其他工作的結果。
            '''
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)

            '''
            Execute a fixed local generation cell, never user-supplied code.
            僅執行 Notebook 內固定的生成程式，不會執行請求傳來的原始碼。
            '''
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):

                '''
                Include YuE2 replay metadata and score with the single MP3.
                將 YuE2 的重播參數與 ABC 樂譜一起提供下載。
                '''
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

'''
Keep one worker and one maintenance thread even after rerunning this cell.
重跑 Cell 仍僅保留一個 worker 和一個清理 thread。
'''
if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()
if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

'''
Close only the previous ngrok tunnel; preserve the running API and its jobs.
只關閉舊的 ngrok 通道，保留正在運作的 API 與所有生成工作。
'''
old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)
NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(Path('/kaggle/working/ngrok'))
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

'''
Always request the auth token again when this cell is rerun.
每次重跑均重新要求輸入 ngrok Authtoken，不顯示內容。
'''
ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')
result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')
del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

'''
Reuse the same API server when this cell is executed again.
重跑本格時沿用同一個 API Server，不中斷生成工作或清空佇列。
'''
server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()
if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:

    '''
    Update the handler on rerun without restarting an active GPU job.
    重跑後更新 API handler，但不重新啟動正在執行的生成工作。
    '''
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')
with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)
if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])
print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')


## 8. Web API 使用說明

In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request

'''
Request the API schema directly from the local API server.
直接從本機 API Server 查詢支援的參數與模式。
'''
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)
print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本地 requests 範例：06_YuE2_音樂生成_Kaggle_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')